<a href="https://colab.research.google.com/github/dang710206/Baitap-AI-/blob/main/Bt11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#BT11
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import folium
from folium.plugins import HeatMap

# Tạo dữ liệu mẫu
np.random.seed(42)

districts = ['Quận 1', 'Quận 3', 'Quận 7', 'Bình Thạnh', 'Gò Vấp', 'Thủ Đức', 'Tân Bình', 'Phú Nhuận']

data = []
for _ in range(5000):
    district = np.random.choice(districts)
    hour = np.random.randint(0, 24)
    day_of_week = np.random.randint(0, 7)
    pop_density = {'Quận 1': 18000, 'Quận 3': 22000, 'Quận 7': 8000, 'Bình Thạnh': 15000,
                   'Gò Vấp': 12000, 'Thủ Đức': 6000, 'Tân Bình': 14000, 'Phú Nhuận': 16000}[district]
    is_rain = np.random.choice([0, 1], p=[0.7, 0.3])
    is_weekend = 1 if day_of_week >= 5 else 0

    base = pop_density / 1000 + (20 if 7 <= hour <= 9 or 17 <= hour <= 20 else 5)
    demand = int(base * (1.5 if is_weekend else 1) * (0.7 if is_rain else 1) + np.random.normal(0, 8))
    demand = max(5, demand)

    data.append([district, hour, day_of_week, is_rain, is_weekend, pop_density, demand])

df = pd.DataFrame(data, columns=['district', 'hour', 'day_of_week', 'is_rain',
                                 'is_weekend', 'pop_density', 'demand'])

# Encoding
df_encoded = pd.get_dummies(df, columns=['district'], drop_first=True)

X = df_encoded.drop('demand', axis=1)
y = df_encoded['demand']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Đánh giá
y_pred = model.predict(X_test)
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred):.2f} chuyến")
print(f"R² Score: {r2_score(y_test, y_pred):.3f}")

# Dự đoán trên bản đồ
district_coords = {
    'Quận 1': [10.7769, 106.7009], 'Quận 3': [10.7828, 106.6950],
    'Quận 7': [10.7295, 106.7218], 'Bình Thạnh': [10.8122, 106.7180],
    'Gò Vấp': [10.8231, 106.6297], 'Thủ Đức': [10.8012, 106.7114],
    'Tân Bình': [10.7962, 106.6635], 'Phú Nhuận': [10.8000, 106.6800]
}

# Tạo mẫu dự đoán cho giờ 18h (cao điểm)
pred_data = []
base_sample = {
    'hour': 18,
    'day_of_week': 3,
    'is_rain': 0,
    'is_weekend': 0,
}

for dist in districts:
    sample = base_sample.copy()
    sample['pop_density'] = {'Quận 1':18000, 'Quận 3':22000, 'Quận 7':8000, 'Bình Thạnh':15000,
                             'Gò Vấp':12000, 'Thủ Đức':6000, 'Tân Bình':14000, 'Phú Nhuận':16000}[dist]

    # Tạo DataFrame và khớp chính xác các cột với X_train
    pred_input = pd.DataFrame([sample])
    pred_input = pd.get_dummies(pred_input, columns=['district'] if 'district' in pred_input else [], drop_first=True)

    # Quan trọng: Khớp lại tất cả cột theo thứ tự của mô hình
    pred_input = pred_input.reindex(columns=X.columns, fill_value=0)

    pred_demand = model.predict(pred_input)[0]
    lat, lon = district_coords[dist]
    pred_data.append([lat, lon, pred_demand / 100])   # scale cho heatmap

# Vẽ bản đồ
m = folium.Map(location=[10.7769, 106.7009], zoom_start=12, tiles="cartodb positron")

HeatMap(pred_data, radius=25, blur=15, max_zoom=13,
        gradient={0.2: 'blue', 0.5: 'green', 0.8: 'yellow', 1.0: 'red'}).add_to(m)

for i, dist in enumerate(districts):
    lat, lon = district_coords[dist]
    demand_val = pred_data[i][2] * 100
    folium.Marker(
        location=[lat, lon],
        popup=f"<b>{dist}</b><br>Nhu cầu dự đoán (18h): {demand_val:.0f} chuyến/giờ",
        icon=folium.Icon(color='red' if demand_val > 60 else 'orange' if demand_val > 40 else 'blue')
    ).add_to(m)

m.save("ban_do_du_doan_nhu_cau_23_11.html")
print("✅ Bản đồ đã tạo thành công: ban_do_du_doan_nhu_cau_23_11.html")
m

Mean Absolute Error: 6.97 chuyến
R² Score: 0.523
✅ Bản đồ đã tạo thành công: ban_do_du_doan_nhu_cau_23_11.html
